# Modül 01: Sinir Ağları Temelleri
## Deep Learning Path — Kapsamlı Jupyter Notebook

> **Bu notebook**, yapay sinir ağlarının temel kavramlarını sıfırdan öğretir.  
> Teori, matematik, görselleştirme ve kod bir arada sunulmuştur.

---

## 📋 İçindekiler

1. [Giriş ve Öğrenme Hedefleri](#1)
2. [Matematiksel Arka Plan](#2)
3. [Biyolojik → Yapay Nöron](#3)
4. [Aktivasyon Fonksiyonları (Görsel)](#4)
5. [XOR Problemi: Neden Tek Katman Yetmez?](#5)
6. [NumPy ile MLP — Sıfırdan](#6)
7. [TensorFlow/Keras İmplementasyonu](#7)
8. [PyTorch İmplementasyonu](#8)
9. [Framework Karşılaştırması](#9)
10. [Hiperparametre Deneyleri](#10)
11. [Özet ve Alıştırmalar](#11)


## 1. Giriş ve Öğrenme Hedefleri <a id='1'></a>

Bu modülü tamamladığında şunları yapabileceksin:

- ✅ Yapay bir nöronun matematiksel modelini açıklamak: $z = \sum w_i x_i + b$, $a = f(z)$
- ✅ İleri beslemeyi (forward pass) matris işlemleriyle hesaplamak
- ✅ XOR problemini çözmek için neden gizli katman gerektiğini kanıtlamak
- ✅ Binary Cross-Entropy kaybını hesaplamak ve yorumlamak
- ✅ Backpropagation'ı zincir kuralıyla adım adım uygulamak
- ✅ Aynı ağı NumPy, TensorFlow ve PyTorch ile kodlamak
- ✅ Öğrenme hızı ve epoch sayısının eğitime etkisini gözlemlemek


## 2. Matematiksel Arka Plan <a id='2'></a>

### 2.1 Tek Nöron Modeli

Bir yapay nöron şu adımları gerçekleştirir:

**Adım 1 — Ağırlıklı Toplam (Weighted Sum):**
$$z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$$

**Adım 2 — Aktivasyon Fonksiyonu:**
$$a = f(z)$$

### 2.2 Katmanlı Yapı (Matris Formu)

$L$ gizli nöronlu bir katman için:

$$\mathbf{z}^{(1)} = \mathbf{X} \mathbf{W}^{(1)} + \mathbf{b}^{(1)}$$
$$\mathbf{a}^{(1)} = \sigma(\mathbf{z}^{(1)})$$
$$\mathbf{z}^{(2)} = \mathbf{a}^{(1)} \mathbf{W}^{(2)} + \mathbf{b}^{(2)}$$
$$\hat{y} = \sigma(\mathbf{z}^{(2)})$$

### 2.3 Binary Cross-Entropy Kaybı

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

### 2.4 Gradient Descent Güncellemesi

$$W \leftarrow W - \eta \cdot \frac{\partial \mathcal{L}}{\partial W}$$

Burada $\eta$ öğrenme hızıdır (learning rate).


## 3. Kurulum ve Veri Hazırlığı <a id='3'></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Grafik stilini ayarla
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

np.random.seed(42)
print("✓ Kütüphaneler yüklendi")
print(f"  NumPy: {np.__version__}")


### 3.1 Biyolojik → Yapay Nöron

```
BİYOLOJİK NÖRON            YAPAY NÖRON
─────────────────           ─────────────
Dendrit (giriş)    →        x₁, x₂, ..., xₙ
Sinaptik ağırlık   →        w₁, w₂, ..., wₙ
Hücre gövdesi     →        z = Σwᵢxᵢ + b
Akson hillock      →        Aktivasyon eşiği
Akson (çıkış)      →        a = f(z)
```


In [ ]:
# Tek nöron: adım adım hesaplama
x = np.array([0.5, -1.2, 0.8])   # Giriş özellikler
w = np.array([0.4, -0.3, 0.7])   # Ağırlıklar (öğrenilecek)
b = 0.1                            # Bias (eşik)

# Adım 1: Ağırlıklı toplam
z = np.dot(w, x) + b

# Adım 2: Sigmoid aktivasyon
sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -500, 500)))
a = sigmoid(z)

print("=" * 45)
print("  TEK NÖRON HESAPLAMA")
print("=" * 45)
print(f"  Giriş x       : {x}")
print(f"  Ağırlıklar w  : {w}")
print(f"  Bias b        : {b}")
print(f"  z = w·x + b   = {0.4*0.5} + {-0.3*-1.2} + {0.7*0.8} + {b}")
print(f"               = {z:.4f}")
print(f"  a = σ({z:.4f}) = {a:.4f}")
print(f"  Yorum: Bu nöron {a:.1%} olasılıkla aktif")


## 4. Aktivasyon Fonksiyonları <a id='4'></a>

Aktivasyon fonksiyonları ağa **doğrusal olmayan** (non-linear) kapasite kazandırır.  
Aktivasyonsuz bir ağ ne kadar derin olursa olsun sadece doğrusal dönüşüm yapar.

| Fonksiyon | Formül | Aralık | Avantaj | Dezavantaj |
|-----------|--------|--------|---------|------------|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | (0,1) | Olasılık çıktısı | Vanishing gradient |
| Tanh | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | (-1,1) | Sıfır merkezli | Hâlâ vanishing |
| ReLU | $\max(0, z)$ | $[0, \infty)$ | Hızlı, basit | Dead neuron |


In [ ]:
# Aktivasyon fonksiyonları ve türevleri
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(a):
    """Sigmoid türevi: σ'(z) = a*(1-a) — a = sigmoid(z)"""
    return a * (1 - a)

def tanh_fn(z):
    return np.tanh(z)

def tanh_deriv(a):
    """tanh türevi: 1 - a²"""
    return 1 - a**2

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    """ReLU türevi: z>0 ise 1, değilse 0"""
    return (z > 0).astype(float)

z = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sol: Aktivasyon fonksiyonları
ax = axes[0]
ax.plot(z, sigmoid(z), label='Sigmoid σ(z)', color='steelblue', lw=2.5)
ax.plot(z, tanh_fn(z), label='Tanh(z)', color='darkorange', lw=2.5)
ax.plot(z, relu(z), label='ReLU(z)', color='crimson', lw=2.5)
ax.axhline(0, color='gray', lw=0.8, alpha=0.5)
ax.axvline(0, color='gray', lw=0.8, alpha=0.5)
ax.set_title('Aktivasyon Fonksiyonları f(z)', fontweight='bold')
ax.set_xlabel('z (ağırlıklı toplam)')
ax.set_ylabel('f(z)')
ax.legend()
ax.set_ylim(-1.5, 2.5)

# Sağ: Türevler (gradyanlar)
sig_vals = sigmoid(z)
tanh_vals = tanh_fn(z)
ax2 = axes[1]
ax2.plot(z, sigmoid_deriv(sig_vals), label="Sigmoid'in türevi", color='steelblue', lw=2.5)
ax2.plot(z, tanh_deriv(tanh_vals), label="Tanh'ın türevi", color='darkorange', lw=2.5)
ax2.plot(z, relu_deriv(z), label="ReLU'nun türevi", color='crimson', lw=2.5)
ax2.axhline(0, color='gray', lw=0.8, alpha=0.5)
ax2.set_title("Türevler f'(z) — Gradyan Büyüklüğü", fontweight='bold')
ax2.set_xlabel('z')
ax2.set_ylabel("f'(z)")
ax2.legend()
ax2.set_ylim(-0.1, 1.2)

plt.tight_layout()
plt.savefig('modul01_aktivasyon.png', dpi=120, bbox_inches='tight')
plt.show()
print("⚠️  Sigmoid türevi z→±∞ iken 0'a yaklaşıyor → Vanishing Gradient!")
print("✓  ReLU türevi sabit (0 veya 1) → Vanishing gradient yok")


## 5. XOR Problemi: Neden Tek Katman Yetmez? <a id='5'></a>

XOR problemi, tek katmanlı ağın yetersizliğini gösteren klasik örnektir.

```
x1  x2  |  XOR
---------+-----
 0   0  |   0   ← sınıf A
 0   1  |   1   ← sınıf B  
 1   0  |   1   ← sınıf B
 1   1  |   0   ← sınıf A
```

**Tek bir doğru çizgi bu iki sınıfı ayıramaz!**  
→ Gizli katman eklersek problem çözülür.


In [ ]:
# XOR: Görsel kanıt — neden lineer ayrılamaz?
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sol: XOR dağılımı — doğrusal ayrılmaz
ax1 = axes[0]
colors = ['royalblue' if y==0 else 'crimson' for y in y_xor.flatten()]
for i, (pt, c) in enumerate(zip(X_xor, colors)):
    ax1.scatter(pt[0], pt[1], c=c, s=250, zorder=5, edgecolors='white', lw=2)
    ax1.annotate(f"  ({int(pt[0])},{int(pt[1])})\nXOR={int(y_xor[i,0])}",
                 (pt[0], pt[1]), fontsize=9)

# Hiçbir doğru bu iki grubu ayıramaz — bunu göster
x_line = np.linspace(-0.3, 1.3, 100)
ax1.plot(x_line, x_line, 'g--', alpha=0.5, label='Deneme 1')
ax1.plot(x_line, 1-x_line, 'm--', alpha=0.5, label='Deneme 2')
ax1.set_title('XOR — Doğrusal Ayrılamaz\n(Tek katman yetersiz)', fontweight='bold')
ax1.set_xlim(-0.5, 1.5); ax1.set_ylim(-0.5, 1.5)
ax1.set_xlabel('x1'); ax1.set_ylabel('x2')
patch0 = mpatches.Patch(color='royalblue', label='XOR=0')
patch1 = mpatches.Patch(color='crimson', label='XOR=1')
ax1.legend(handles=[patch0, patch1])
ax1.text(0.5, -0.4, '⚠️ Hiçbir doğru 0 ve 1 sınıflarını ayıramaz!',
         ha='center', fontsize=9, color='red')

# Sağ: MLP çözümü konsepti
ax2 = axes[1]
ax2.text(0.5, 0.85, 'MLP Gizli Katman Dönüşümü', ha='center', fontsize=12,
         fontweight='bold', transform=ax2.transAxes)
ax2.text(0.5, 0.65, 'Gizli katman uzayı dönüştürür:
(0,0)→(0,0)  (1,1)→(0,0)
(0,1)→(1,0)  (1,0)→(0,1)',
         ha='center', fontsize=10, transform=ax2.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax2.text(0.5, 0.35, '✓ Yeni uzayda doğrusal ayrılabilir!
→ Tek bir çizgi yeterli',
         ha='center', fontsize=10, color='green', transform=ax2.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax2.axis('off')

plt.tight_layout()
plt.show()
print("Sonuç: Gizli katman veriyi yeni bir uzaya taşır → lineer ayrılabilir hale getirir")


## 6. NumPy ile MLP — Sıfırdan <a id='6'></a>

In [ ]:
class NumpyMLP:
    """Çok Katmanlı Perceptron — Sıfırdan NumPy ile"""
    
    def __init__(self, input_size, hidden_size, output_size, lr=0.5):
        self.lr = lr
        # Xavier başlatma
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2/(input_size+hidden_size))
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2/(hidden_size+output_size))
        self.b2 = np.zeros((1, output_size))
        self.loss_history = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self._sigmoid(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self._sigmoid(self.z2)
        return self.a2

    def loss(self, y, yhat):
        eps = 1e-15
        yhat = np.clip(yhat, eps, 1-eps)
        return -np.mean(y * np.log(yhat) + (1-y) * np.log(1-yhat))

    def backward(self, X, y):
        n = X.shape[0]
        dL = (self.a2 - y) / n
        dW2 = self.a1.T @ dL
        db2 = np.sum(dL, axis=0, keepdims=True)
        dL2 = dL @ self.W2.T * (self.a1 * (1-self.a1))
        dW1 = X.T @ dL2
        db1 = np.sum(dL2, axis=0, keepdims=True)
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1

    def train(self, X, y, epochs=10000):
        for _ in range(epochs):
            yhat = self.forward(X)
            self.loss_history.append(self.loss(y, yhat))
            self.backward(X, y)
    
    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int)

# Eğit
np.random.seed(42)
mlp = NumpyMLP(2, 4, 1, lr=0.5)
mlp.train(X_xor, y_xor, epochs=10000)

# Sonuç
preds = mlp.predict(X_xor)
probs = mlp.forward(X_xor)
acc = np.mean(preds == y_xor)

print(f"{'x1':>4} {'x2':>4} | {'Beklenen':>10} | {'Tahmin':>8} | {'P(y=1)':>8}")
print("-" * 45)
for i in range(4):
    ok = '✓' if preds[i,0]==int(y_xor[i,0]) else '✗'
    print(f"{int(X_xor[i,0]):>4} {int(X_xor[i,1]):>4} | {int(y_xor[i,0]):>10} | {preds[i,0]:>8} | {probs[i,0]:>7.4f} {ok}")
print(f"\n✓ Doğruluk: {acc:.1%}")


In [ ]:
# Karar sınırı görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: Karar sınırı
ax1 = axes[0]
xx, yy = np.meshgrid(np.linspace(-0.5,1.5,200), np.linspace(-0.5,1.5,200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs_grid = mlp.forward(grid).reshape(xx.shape)
ax1.contourf(xx, yy, probs_grid, levels=50, cmap='RdYlBu', alpha=0.7)
ax1.contour(xx, yy, probs_grid, levels=[0.5], colors='k', lw=2, linestyles='--')
colors_d = ['royalblue' if y==0 else 'crimson' for y in y_xor.flatten()]
for pt, c in zip(X_xor, colors_d):
    ax1.scatter(pt[0], pt[1], c=c, s=250, zorder=5, edgecolors='white', lw=2)
ax1.set_title('MLP Karar Sınırı\n(XOR problemi çözüldü!)', fontweight='bold')
ax1.set_xlabel('x1'); ax1.set_ylabel('x2')

# Sağ: Loss eğrisi
ax2 = axes[1]
ax2.plot(mlp.loss_history, color='steelblue', lw=2)
ax2.set_yscale('log')
ax2.set_title('Eğitim Loss Eğrisi (log ölçek)', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Binary Cross-Entropy Loss')
ax2.axhline(y=0.01, color='red', linestyle='--', alpha=0.5, label='Loss=0.01')
ax2.legend()

plt.tight_layout()
plt.show()


## 10. Hiperparametre Deneyleri <a id='10'></a>

In [ ]:
# Farklı öğrenme hızlarının etkisi
learning_rates = [0.01, 0.1, 0.5, 1.0, 5.0]
results = {}

for lr in learning_rates:
    np.random.seed(42)
    model = NumpyMLP(2, 4, 1, lr=lr)
    model.train(X_xor, y_xor, epochs=5000)
    final_acc = np.mean(model.predict(X_xor) == y_xor)
    results[lr] = {'loss': model.loss_history, 'acc': final_acc}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors_lr = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']

ax1 = axes[0]
for (lr, res), c in zip(results.items(), colors_lr):
    ax1.plot(res['loss'], label=f'lr={lr}', color=c, lw=2, alpha=0.85)
ax1.set_yscale('log')
ax1.set_title('Öğrenme Hızı Karşılaştırması', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss (log)')
ax1.legend(); ax1.set_xlim(0, 5000)

ax2 = axes[1]
bars = ax2.bar([str(lr) for lr in learning_rates],
               [r['acc'] for r in results.values()],
               color=colors_lr, edgecolor='white', linewidth=1.2)
ax2.set_title('Final Doğruluk vs Öğrenme Hızı', fontweight='bold')
ax2.set_xlabel('Öğrenme Hızı (lr)'); ax2.set_ylabel('Doğruluk')
ax2.set_ylim(0, 1.1)
for bar, (lr, res) in zip(bars, results.items()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{res['acc']:.0%}", ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nGözlem:")
print("  lr çok küçük → yavaş öğrenir")
print("  lr çok büyük → salınım/ıraksama (diverge)")
print("  Optimal lr aralığı: ~0.1–1.0 (bu probleme özgü)")


## 11. Özet, Alıştırmalar ve Sonraki Adım <a id='11'></a>

### 📌 Bu Modülden Öğrendiklerimiz

| Kavram | Formül / Kural | Ne Zaman Kullanılır |
|--------|---------------|---------------------|
| Yapay nöron | $z=\mathbf{w}^T\mathbf{x}+b$, $a=f(z)$ | Her katmanda |
| Sigmoid | $\sigma(z)=1/(1+e^{-z})$ | Çıkış katmanı (ikili) |
| BCE Loss | $-rac{1}{n}\sum[y\log\hat{y}+(1-y)\log(1-\hat{y})]$ | İkili sınıflandırma |
| Gradient Descent | $W \leftarrow W - \eta 
abla_W \mathcal{L}$ | Her epoch |
| Xavier Init | $W \sim \mathcal{N}(0, rac{2}{n_{in}+n_{out}})$ | Ağırlık başlatma |

---

### 🧪 Alıştırmalar

**1.** `NumpyMLP` sınıfına `ReLU` aktivasyonunu ekle ve sonuçları karşılaştır.  
*(İpucu: `relu(z) = np.maximum(0, z)` ve türevi `(z>0).astype(float)`)*

**2.** Gizli katman nöron sayısını 2, 4, 8, 16 olarak değiştir. Her durumda kaç epoch'ta %100 doğruluğa ulaşılıyor?

**3.** XOR yerine AND veya OR veri seti oluştur. Tek katmanlı ağ (hidden_size=0) çözebiliyor mu? Neden?

**4.** `compute_loss` fonksiyonunu MSE (Mean Squared Error) kullanacak şekilde değiştir: $\mathcal{L} = rac{1}{n}\sum(y-\hat{y})^2$. Convergence hızı nasıl değişiyor?

**5.** Eğitim sırasında her 100 epoch'ta ağırlık histogramını kaydet ve animasyon oluştur.

---

### 🚀 Sonraki Modül: Aktivasyon Fonksiyonları

Modül 02'de şunları öğreneceğiz:
- Vanishing gradient probleminin matematiksel kanıtı
- Modern aktivasyonlar: ELU, SELU, Swish, GELU
- Hangi aktivasyonu ne zaman seçmeli? Karar rehberi
